In [1]:
import torch
import numpy as np
import pandas as pd

In [2]:
import numpy as np
import pandas as pd
import torch
import os

import seaborn as sns
import matplotlib.pyplot as plt
import ot
import re

In [5]:
def drop_unlabeled(df):
    """
    Remove rows and columns labeled as 'Unlabeled' from a square pandas DataFrame.
    """
    return df.loc[~df.index.isin(['unassigned']), ~df.columns.isin(['unassigned'])]

In [6]:
def rename_cell_type(cell_type):
    return re.sub(r'[^a-zA-Z0-9\/+&]', '_', cell_type)

def aggregate_attention_scores(attention_scores, cell_type_matrix):
    """Compute the sum of attention scores for each sender-receiver cell type pair 
    and normalize by the number of receiver cells of each type, ensuring float values."""
    
    # Extract receiver and sender cell types
    receiver_types = cell_type_matrix[:, 0]  # First column: receiver cell type
    sender_types = cell_type_matrix[:, 1:]  # Other columns: sender cell type

    # Flatten data to count interactions
    data = []
    for i in range(attention_scores.shape[0]):
        for j in range(attention_scores.shape[1]):
            data.append((receiver_types[i], sender_types[i, j], float(attention_scores[i, j])))

    df = pd.DataFrame(data, columns=["Receiver", "Sender", "Score"])

    # Compute sum of attention scores for each sender-receiver pair
    sum_scores = df.groupby(["Sender", "Receiver"])["Score"].sum().unstack(fill_value=0)

    # Count the number of receiver cells per receiver type
    receiver_counts = df.groupby("Receiver").size()

    # Normalize by the number of receiver cells per receiver type
    normalized_df = sum_scores.div(receiver_counts, axis=1).fillna(0)

    # Rename cell types
    normalized_df.index = [rename_cell_type(idx) for idx in normalized_df.index]
    normalized_df.columns = [rename_cell_type(col) for col in normalized_df.columns]

    # Aggregate after renaming
    normalized_df = normalized_df.groupby(normalized_df.index).sum()
    normalized_df = normalized_df.groupby(normalized_df.columns, axis=1).sum()
    
    # Sort the dataframe
    sorted_index = sorted(normalized_df.index)
    sorted_columns = sorted(normalized_df.columns)
    normalized_df = normalized_df.loc[sorted_index, sorted_columns]

    # Convert all values to float explicitly
    normalized_df = normalized_df.astype(float)

    return normalized_df

In [7]:
method="BIDCell"

file_path="../"+method+"/influence_tensor/edges_sample01.pth"
results=torch.load(file_path,weights_only=False)

cell_type_matrix=np.array(results['cell_type_name'])
print(np.unique(cell_type_matrix[:,0]))

attention_scores=results["attention_score"]/8
attention_scores=torch.abs(attention_scores)
attention_scores=attention_scores/torch.sum(attention_scores,dim=(0,1),keepdim=True)
attention_scores=torch.mean(attention_scores,dim=-1)
#attention_scores=attention_scores/torch.sum(attention_scores,dim=-1,keepdim=True)
print(attention_scores.shape,attention_scores)

GITIII=aggregate_attention_scores(attention_scores, cell_type_matrix)
GITIII=drop_unlabeled(GITIII)
print(GITIII)

GITIII.to_csv("./overall_strength/"+method+".csv")

['ACTA2+ Myoepi' 'B Cells' 'CD163+ Macrophage' 'CD4 T' 'CD8 T'
 'CRABP2+ Malignant' 'ECM1+ Malignant' 'Fibroblast' 'IRF7+ DC'
 'KRT15+ Myoepi' 'LAMP3+ DC' 'Macrophage' 'Mast cells' 'Plasma'
 'SCGB2A2+ Malignant' 'STAB2+ Endothelial' 'VWF+ Endothelial' 'unassigned']
torch.Size([103198, 49]) tensor([[1.0680e-06, 5.4780e-07, 6.9432e-07,  ..., 9.9660e-08, 8.1765e-08,
         3.7312e-07],
        [2.5465e-06, 4.0699e-07, 3.2530e-07,  ..., 3.2037e-07, 1.4343e-07,
         1.0989e-07],
        [6.2063e-07, 4.2750e-07, 1.8544e-07,  ..., 2.2185e-07, 9.0975e-08,
         1.5490e-07],
        ...,
        [1.5371e-06, 6.4042e-07, 6.4373e-07,  ..., 1.1174e-07, 2.3274e-07,
         1.0489e-07],
        [8.2634e-07, 3.9814e-07, 7.8560e-07,  ..., 1.0268e-07, 1.2699e-07,
         1.1312e-07],
        [4.9031e-07, 1.8144e-07, 1.5297e-07,  ..., 3.5519e-08, 3.5393e-08,
         3.5316e-08]])
                    ACTA2+_Myoepi       B_Cells  CD163+_Macrophage  \
ACTA2+_Myoepi        3.439732e-08  1.229272

/tmp/ipykernel_3388616/3096831869.py:35: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  normalized_df = normalized_df.groupby(normalized_df.columns, axis=1).sum()


In [8]:
method="v"

file_path="../"+method+"/influence_tensor/edges_sample01.pth"
results=torch.load(file_path,weights_only=False)

cell_type_matrix=np.array(results['cell_type_name'])
print(np.unique(cell_type_matrix[:,0]))

attention_scores=results["attention_score"]/8
attention_scores=torch.abs(attention_scores)
attention_scores=attention_scores/torch.sum(attention_scores,dim=(0,1),keepdim=True)
attention_scores=torch.mean(attention_scores,dim=-1)
#attention_scores=attention_scores/torch.sum(attention_scores,dim=-1,keepdim=True)
print(attention_scores.shape,attention_scores)

GITIII=aggregate_attention_scores(attention_scores, cell_type_matrix)
GITIII=drop_unlabeled(GITIII)
print(GITIII)

GITIII.to_csv("./overall_strength/"+method+".csv")

['ACTA2+ Myoepi' 'B Cells' 'CD163+ Macrophage' 'CD4 T' 'CD8 T'
 'CRABP2+ Malignant' 'ECM1+ Malignant' 'Fibroblast' 'IRF7+ DC'
 'KRT15+ Myoepi' 'LAMP3+ DC' 'Macrophage' 'Mast cells' 'Plasma'
 'SCGB2A2+ Malignant' 'STAB2+ Endothelial' 'VWF+ Endothelial' 'unassigned']
torch.Size([106224, 49]) tensor([[1.1286e-06, 1.9073e-06, 7.5703e-07,  ..., 1.1915e-07, 9.5239e-08,
         1.3303e-07],
        [1.7343e-06, 2.1695e-06, 2.1981e-07,  ..., 2.0575e-07, 4.1505e-08,
         5.3551e-08],
        [4.1563e-06, 9.8567e-07, 1.3437e-06,  ..., 5.6197e-08, 8.2372e-08,
         1.0179e-07],
        ...,
        [1.2143e-06, 5.6584e-07, 9.2597e-07,  ..., 7.1430e-08, 3.4515e-08,
         5.6560e-08],
        [6.7406e-07, 5.4115e-07, 8.7710e-07,  ..., 7.4140e-08, 3.5656e-08,
         1.2022e-07],
        [4.3825e-06, 4.1342e-07, 2.1809e-07,  ..., 9.1450e-08, 6.4882e-08,
         2.6954e-08]])


/tmp/ipykernel_3388616/3096831869.py:35: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  normalized_df = normalized_df.groupby(normalized_df.columns, axis=1).sum()


                    ACTA2+_Myoepi       B_Cells  CD163+_Macrophage  \
ACTA2+_Myoepi        3.195776e-08  6.784098e-10       1.952517e-09   
B_Cells              2.483610e-10  1.089726e-08       4.622381e-09   
CD163+_Macrophage    2.930136e-09  1.934377e-08       1.753901e-08   
CD4_T                7.048623e-09  8.874313e-08       3.786592e-08   
CD8_T                8.936772e-09  2.492180e-08       1.291022e-08   
CRABP2+_Malignant    5.609347e-08  7.008975e-10       1.318269e-09   
ECM1+_Malignant      1.719713e-09  1.170252e-09       3.682215e-09   
Fibroblast           3.161688e-08  2.290258e-08       7.514741e-08   
IRF7+_DC             2.130409e-10  4.167753e-09       2.661089e-09   
KRT15+_Myoepi        2.143379e-08  4.054604e-10       2.203821e-10   
LAMP3+_DC            6.845556e-11  6.216404e-10       2.216217e-10   
Macrophage           2.184321e-08  3.271519e-08       3.060981e-08   
Mast_cells           2.212447e-10  5.673854e-10       8.605810e-10   
Plasma              

In [9]:
method="JSTA"

file_path="../"+method+"/influence_tensor/edges_sample01.pth"
results=torch.load(file_path,weights_only=False)

cell_type_matrix=np.array(results['cell_type_name'])
print(np.unique(cell_type_matrix[:,0]))

attention_scores=results["attention_score"]/8
attention_scores=torch.abs(attention_scores)
attention_scores=attention_scores/torch.sum(attention_scores,dim=(0,1),keepdim=True)
attention_scores=torch.mean(attention_scores,dim=-1)
#attention_scores=attention_scores/torch.sum(attention_scores,dim=-1,keepdim=True)
print(attention_scores.shape,attention_scores)

GITIII=aggregate_attention_scores(attention_scores, cell_type_matrix)
GITIII=drop_unlabeled(GITIII)
print(GITIII)

GITIII.to_csv("./overall_strength/"+method+".csv")

['ACTA2+ Myoepi' 'B Cells' 'CD163+ Macrophage' 'CD4 T' 'CD8 T'
 'CRABP2+ Malignant' 'ECM1+ Malignant' 'Fibroblast' 'IRF7+ DC'
 'KRT15+ Myoepi' 'LAMP3+ DC' 'Macrophage' 'Mast cells' 'Plasma'
 'SCGB2A2+ Malignant' 'STAB2+ Endothelial' 'VWF+ Endothelial' 'unassigned']
torch.Size([107122, 49]) tensor([[4.9767e-06, 1.0441e-06, 1.5263e-07,  ..., 5.3506e-08, 1.3634e-07,
         4.9136e-08],
        [7.3498e-07, 2.7356e-07, 2.2254e-07,  ..., 2.9372e-08, 1.2680e-07,
         2.9101e-08],
        [1.7994e-06, 1.1002e-06, 1.2018e-06,  ..., 5.4592e-08, 1.0387e-07,
         7.8949e-08],
        ...,
        [1.8994e-07, 1.8912e-07, 1.1634e-06,  ..., 2.6971e-07, 2.1871e-07,
         2.0935e-07],
        [5.9099e-07, 4.2557e-07, 5.6463e-07,  ..., 5.6959e-08, 1.4467e-07,
         1.3252e-07],
        [1.7879e-06, 1.9075e-07, 1.2144e-06,  ..., 1.4954e-07, 1.0628e-07,
         2.7517e-07]])


/tmp/ipykernel_3388616/3096831869.py:35: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  normalized_df = normalized_df.groupby(normalized_df.columns, axis=1).sum()


                    ACTA2+_Myoepi       B_Cells  CD163+_Macrophage  \
ACTA2+_Myoepi        2.898274e-08  7.561330e-10       1.902958e-09   
B_Cells              3.948896e-10  1.453585e-08       5.411002e-09   
CD163+_Macrophage    2.636929e-09  1.366862e-08       1.667578e-08   
CD4_T                8.439809e-09  8.556001e-08       4.053956e-08   
CD8_T                9.497344e-09  2.261031e-08       1.437592e-08   
CRABP2+_Malignant    4.843585e-08  4.566601e-10       1.114013e-09   
ECM1+_Malignant      1.465463e-09  1.572343e-09       4.652917e-09   
Fibroblast           3.135580e-08  2.873340e-08       7.315081e-08   
IRF7+_DC             1.155737e-10  4.129169e-09       2.564278e-09   
KRT15+_Myoepi        1.958089e-08  2.130685e-10       2.326003e-10   
LAMP3+_DC            1.458029e-10  9.586941e-10       2.229213e-10   
Macrophage           2.173545e-08  2.760921e-08       2.787773e-08   
Mast_cells           2.509553e-10  6.199089e-10       1.372893e-09   
Plasma              